In [2]:
from dotenv import load_dotenv
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob as gl
import scipy.optimize
from tqdm.auto import tqdm
from scipy.interpolate import InterpolatedUnivariateSpline
from scipy.spatial import distance
import pymap3d as pm
import pymap3d.vincenty as pmv

# Constants
CLIGHT = 299_792_458   # speed of light (m/s)
RE_WGS84 = 6_378_137   # earth semimajor axis (WGS84) (m)
OMGE = 7.2921151467E-5  # earth angular velocity (IS-GPS) (rad/s)

In [2]:
parent_folder = "../test"

# retrieve device.imu files from train
imu_test_files = gl.glob(os.path.join(parent_folder, "*", "*", "device_imu.csv"))
print(imu_test_files) 

['../test/2022-02-23-US-LAX-3/XiaomiMi8/device_imu.csv', '../test/2021-09-14-US-MTV-1/GooglePixel5/device_imu.csv', '../test/2021-08-17-US-MTV-1/GooglePixel5/device_imu.csv', '../test/2022-02-23-US-LAX-5/XiaomiMi8/device_imu.csv', '../test/2021-09-20-US-MTV-1/XiaomiMi8/device_imu.csv', '../test/2022-03-14-US-MTV-1/GooglePixel5/device_imu.csv', '../test/2022-04-22-US-OAK-1/GooglePixel5/device_imu.csv', '../test/2022-01-11-US-MTV-1/GooglePixel6Pro/device_imu.csv', '../test/2021-08-24-US-SVL-2/GooglePixel5/device_imu.csv', '../test/2022-04-25-US-OAK-1/GooglePixel5/device_imu.csv', '../test/2022-03-31-US-LAX-3/SamsungGalaxyS20Ultra/device_imu.csv', '../test/2022-02-24-US-LAX-3/XiaomiMi8/device_imu.csv', '../test/2022-01-18-US-SJC-2/GooglePixel5/device_imu.csv', '../test/2022-04-01-US-LAX-3/XiaomiMi8/device_imu.csv', '../test/2022-02-24-US-LAX-5/SamsungGalaxyS20Ultra/device_imu.csv', '../test/2022-02-01-US-SJC-1/XiaomiMi8/device_imu.csv', '../test/2022-03-22-US-MTV-1/SamsungGalaxyS20Ultra/d

In [3]:
imu_list = []

for file in imu_test_files:
    imu_file = pd.read_csv(file)
    parts = file.split(os.sep) # eg ['train', '2021-12-28-US-MTV-1', 'GooglePixel5', 'device_imu.csv']
    imu_file["drive_id"] = parts[-3] 
    imu_file["device"] = parts[-2]
    imu_list.append(imu_file)

# combine all
imu_test_df = pd.concat(imu_list, ignore_index=True)
print(imu_test_df.head())
print()
print(imu_test_df["device"].unique())
print()
print(imu_test_df["drive_id"].unique())

  MessageType  utcTimeMillis  MeasurementX  MeasurementY  MeasurementZ  \
0  UncalAccel  1645655742002     -0.169811      9.894553     -2.316526   
1    UncalMag  1645655742003    -99.040440   -173.611630     -0.707867   
2  UncalAccel  1645655742012     -0.095563      9.671811     -2.009955   
3    UncalMag  1645655742013    -99.040440   -173.611630     -0.707867   
4   UncalGyro  1645655742018     -0.004647     -0.002223     -0.000786   

       BiasX     BiasY    BiasZ             drive_id     device  
0   0.000000    0.0000   0.0000  2022-02-23-US-LAX-3  XiaomiMi8  
1 -83.628395 -180.3668  74.5565  2022-02-23-US-LAX-3  XiaomiMi8  
2   0.000000    0.0000   0.0000  2022-02-23-US-LAX-3  XiaomiMi8  
3 -83.628395 -180.3668  74.5565  2022-02-23-US-LAX-3  XiaomiMi8  
4   0.000000    0.0000   0.0000  2022-02-23-US-LAX-3  XiaomiMi8  

['XiaomiMi8' 'GooglePixel5' 'GooglePixel6Pro' 'SamsungGalaxyS20Ultra'
 'GooglePixel4']

['2022-02-23-US-LAX-3' '2021-09-14-US-MTV-1' '2021-08-17-US-MTV-1'
 '2

In [4]:
imu_testacc_df = imu_test_df[imu_test_df["MessageType"] == "UncalAccel"][["utcTimeMillis", "MeasurementX", "MeasurementY", "MeasurementZ", "drive_id", "device"]]

print(imu_testacc_df.head())

    utcTimeMillis  MeasurementX  MeasurementY  MeasurementZ  \
0   1645655742002     -0.169811      9.894553     -2.316526   
2   1645655742012     -0.095563      9.671811     -2.009955   
5   1645655742022     -0.196157      9.611935     -1.911757   
7   1645655742032     -0.217713      9.770010     -2.201562   
10  1645655742042     -0.153045      9.736478     -2.096178   

               drive_id     device  
0   2022-02-23-US-LAX-3  XiaomiMi8  
2   2022-02-23-US-LAX-3  XiaomiMi8  
5   2022-02-23-US-LAX-3  XiaomiMi8  
7   2022-02-23-US-LAX-3  XiaomiMi8  
10  2022-02-23-US-LAX-3  XiaomiMi8  


In [5]:
imu_testmag_df = pd.DataFrame()

imu_testuncalmag_df = imu_test_df[imu_test_df["MessageType"] == "UncalMag"][["utcTimeMillis", "MeasurementX", "MeasurementY", "MeasurementZ", "BiasX", "BiasY", "BiasZ", "drive_id", "device"]]
imu_testmag_df["utcTimeMillis"] = imu_testuncalmag_df["utcTimeMillis"]
imu_testmag_df["MeasurementX"] = imu_testuncalmag_df["MeasurementX"] + imu_testuncalmag_df["BiasX"]
imu_testmag_df["MeasurementY"] = imu_testuncalmag_df["MeasurementY"] + imu_testuncalmag_df["BiasY"]
imu_testmag_df["MeasurementZ"] = imu_testuncalmag_df["MeasurementZ"] + imu_testuncalmag_df["BiasZ"]
imu_testmag_df["drive_id"] = imu_testuncalmag_df["drive_id"]
imu_testmag_df["device"] = imu_testuncalmag_df["device"]

print(imu_testmag_df.head())

    utcTimeMillis  MeasurementX  MeasurementY  MeasurementZ  \
1   1645655742003   -182.668835    -353.97843     73.848633   
3   1645655742013   -182.668835    -353.97843     73.848633   
6   1645655742023   -184.106645    -354.55757     73.282571   
8   1645655742033   -184.106645    -354.55757     73.282571   
11  1645655742043   -182.623265    -354.81875     71.161257   

               drive_id     device  
1   2022-02-23-US-LAX-3  XiaomiMi8  
3   2022-02-23-US-LAX-3  XiaomiMi8  
6   2022-02-23-US-LAX-3  XiaomiMi8  
8   2022-02-23-US-LAX-3  XiaomiMi8  
11  2022-02-23-US-LAX-3  XiaomiMi8  


In [6]:
imu_testgyro_df = imu_test_df[imu_test_df["MessageType"] == "UncalGyro"][["utcTimeMillis", "MeasurementX", "MeasurementY", "MeasurementZ", "drive_id", "device"]]

print(imu_testgyro_df.head())

    utcTimeMillis  MeasurementX  MeasurementY  MeasurementZ  \
4   1645655742018     -0.004647     -0.002223     -0.000786   
9   1645655742037      0.011067     -0.001690      0.001345   
14  1645655742057      0.006273     -0.001690     -0.000520   
19  1645655742077     -0.003582     -0.002755      0.000013   
24  1645655742097      0.008936     -0.001956      0.000279   

               drive_id     device  
4   2022-02-23-US-LAX-3  XiaomiMi8  
9   2022-02-23-US-LAX-3  XiaomiMi8  
14  2022-02-23-US-LAX-3  XiaomiMi8  
19  2022-02-23-US-LAX-3  XiaomiMi8  
24  2022-02-23-US-LAX-3  XiaomiMi8  


In [7]:
parent_folder = "../test"

# retrieve device.imu files from train
gnss_test_files = gl.glob(os.path.join(parent_folder, "*", "*", "device_gnss.csv"))
print(gnss_test_files) 

['../test/2022-02-23-US-LAX-3/XiaomiMi8/device_gnss.csv', '../test/2021-09-14-US-MTV-1/GooglePixel5/device_gnss.csv', '../test/2021-08-17-US-MTV-1/GooglePixel5/device_gnss.csv', '../test/2022-02-23-US-LAX-5/XiaomiMi8/device_gnss.csv', '../test/2021-09-20-US-MTV-1/XiaomiMi8/device_gnss.csv', '../test/2022-03-14-US-MTV-1/GooglePixel5/device_gnss.csv', '../test/2022-04-22-US-OAK-1/GooglePixel5/device_gnss.csv', '../test/2022-01-11-US-MTV-1/GooglePixel6Pro/device_gnss.csv', '../test/2021-08-24-US-SVL-2/GooglePixel5/device_gnss.csv', '../test/2022-04-25-US-OAK-1/GooglePixel5/device_gnss.csv', '../test/2022-03-31-US-LAX-3/SamsungGalaxyS20Ultra/device_gnss.csv', '../test/2022-02-24-US-LAX-3/XiaomiMi8/device_gnss.csv', '../test/2022-01-18-US-SJC-2/GooglePixel5/device_gnss.csv', '../test/2022-04-01-US-LAX-3/XiaomiMi8/device_gnss.csv', '../test/2022-02-24-US-LAX-5/SamsungGalaxyS20Ultra/device_gnss.csv', '../test/2022-02-01-US-SJC-1/XiaomiMi8/device_gnss.csv', '../test/2022-03-22-US-MTV-1/Samsung

In [8]:
gnss_list = []

for file in gnss_test_files:
    gnss_file = pd.read_csv(file)
    parts = file.split(os.sep) # eg ['train', '2021-12-28-US-MTV-1', 'GooglePixel5', 'device_imu.csv']
    gnss_file["drive_id"] = parts[-3] 
    gnss_file["device"] = parts[-2]
    gnss_list.append(gnss_file)

# combine all
gnss_test_df = pd.concat(gnss_list, ignore_index=True)
print(gnss_test_df.head())
print()
print(gnss_test_df["device"].unique())
print()
print(gnss_test_df["drive_id"].unique())

  MessageType  utcTimeMillis    TimeNanos  LeapSecond        FullBiasNanos  \
0         Raw  1645655742000  74816000000         NaN -1329690885184087331   
1         Raw  1645655742000  74816000000         NaN -1329690885184087331   
2         Raw  1645655742000  74816000000         NaN -1329690885184087331   
3         Raw  1645655742000  74816000000         NaN -1329690885184087331   
4         Raw  1645655742000  74816000000         NaN -1329690885184087331   

   BiasNanos  BiasUncertaintyNanos  DriftNanosPerSecond  \
0        0.0              5.985788                  NaN   
1        0.0              5.985788                  NaN   
2        0.0              5.985788                  NaN   
3        0.0              5.985788                  NaN   
4        0.0              5.985788                  NaN   

   DriftUncertaintyNanosPerSecond  HardwareClockDiscontinuityCount  ...  \
0                             NaN                                0  ...   
1                         

In [9]:
# Satellite selection using carrier frequency error, elevation angle, and C/N0
def satellite_selection(df, column):
    """
    Args:
        df : DataFrame from device_gnss.csv
        column : Column name
    Returns:
        df: DataFrame with eliminated satellite signals
    """
    idx = df[column].notnull()
    idx &= df['CarrierErrorHz'] < 2.0e6  # carrier frequency error (Hz)
    idx &= df['SvElevationDegrees'] > 10.0  # elevation angle (deg)
    idx &= df['Cn0DbHz'] > 15.0  # C/N0 (dB-Hz)
    idx &= df['MultipathIndicator'] == 0 # Multipath flag

    return df[idx]

In [10]:
# Compute line-of-sight vector from user to satellite
def los_vector(xusr, xsat):
    """
    Args:
        xusr : user position in ECEF (m)
        xsat : satellite position in ECEF (m)
    Returns:
        u: unit line-of-sight vector in ECEF (m)
        rng: distance between user and satellite (m)
    """
    u = xsat - xusr
    rng = np.linalg.norm(u, axis=1).reshape(-1, 1)
    u /= rng
    
    return u, rng.reshape(-1)


# Compute Jacobian matrix
def jac_pr_residuals(x, xsat, pr, W):
    """
    Args:
        x : current position in ECEF (m)
        xsat : satellite position in ECEF (m)
        pr : pseudorange (m)
        W : weight matrix
    Returns:
        W*J : Jacobian matrix
    """
    u, _ = los_vector(x[:3], xsat)
    J = np.hstack([-u, np.ones([len(pr), 1])])  # J = [-ux -uy -uz 1]

    return W @ J


# Compute pseudorange residuals
def pr_residuals(x, xsat, pr, W):
    """
    Args:
        x : current position in ECEF (m)
        xsat : satellite position in ECEF (m)
        pr : pseudorange (m)
        W : weight matrix
    Returns:
        residuals*W : pseudorange residuals
    """
    u, rng = los_vector(x[:3], xsat)

    # Approximate correction of the earth rotation (Sagnac effect) often used in GNSS positioning
    rng += OMGE * (xsat[:, 0] * x[1] - xsat[:, 1] * x[0]) / CLIGHT

    # Add GPS L1 clock offset
    residuals = rng - (pr - x[3])

    return residuals @ W


# Compute Jacobian matrix
def jac_prr_residuals(v, vsat, prr, x, xsat, W):
    """
    Args:
        v : current velocity in ECEF (m/s)
        vsat : satellite velocity in ECEF (m/s)
        prr : pseudorange rate (m/s)
        x : current position in ECEF (m)
        xsat : satellite position in ECEF (m)
        W : weight matrix
    Returns:
        W*J : Jacobian matrix
    """
    u, _ = los_vector(x[:3], xsat)
    J = np.hstack([-u, np.ones([len(prr), 1])])

    return W @ J


# Compute pseudorange rate residuals
def prr_residuals(v, vsat, prr, x, xsat, W):
    """
    Args:
        v : current velocity in ECEF (m/s)
        vsat : satellite velocity in ECEF (m/s)
        prr : pseudorange rate (m/s)
        x : current position in ECEF (m)
        xsat : satellite position in ECEF (m)
        W : weight matrix
    Returns:
        residuals*W : pseudorange rate residuals
    """
    u, rng = los_vector(x[:3], xsat)
    rate = np.sum((vsat-v[:3])*u, axis=1) \
          + OMGE / CLIGHT * (vsat[:, 1] * x[0] + xsat[:, 1] * v[0]
                           - vsat[:, 0] * x[1] - xsat[:, 0] * v[1])

    residuals = rate - (prr - v[3])

    return residuals @ W

In [11]:
# Carrier smoothing of pseudarange
def carrier_smoothing(gnss_df):
    """
    Args:
        df : DataFrame from device_gnss.csv
    Returns:
        df: DataFrame with carrier-smoothing pseudorange 'pr_smooth'
    """
    carr_th = 1.5 # carrier phase jump threshold [m] ** 2.0 -> 1.5 **
    pr_th =  20.0 # pseudorange jump threshold [m]

    prsmooth = np.full_like(gnss_df['RawPseudorangeMeters'], np.nan)
    # Loop for each signal
    for (i, (svid_sigtype, df)) in enumerate((gnss_df.groupby(['Svid', 'SignalType']))):
        df = df.replace(
            {'AccumulatedDeltaRangeMeters': {0: np.nan}})  # 0 to NaN

        # Compare time difference between pseudorange/carrier with Doppler
        drng1 = df['AccumulatedDeltaRangeMeters'].diff() - df['PseudorangeRateMetersPerSecond']
        drng2 = df['RawPseudorangeMeters'].diff() - df['PseudorangeRateMetersPerSecond']

        # Check cycle-slip
        slip1 = (df['AccumulatedDeltaRangeState'].to_numpy() & 2**1) != 0  # reset flag
        slip2 = (df['AccumulatedDeltaRangeState'].to_numpy() & 2**2) != 0  # cycle-slip flag
        slip3 = np.fabs(drng1.to_numpy()) > carr_th # Carrier phase jump
        slip4 = np.fabs(drng2.to_numpy()) > pr_th # Pseudorange jump

        idx_slip = slip1 | slip2 | slip3 | slip4
        idx_slip[0] = True

        # groups with continuous carrier phase tracking
        df['group_slip'] = np.cumsum(idx_slip)

        # Psudorange - carrier phase
        df['dpc'] = df['RawPseudorangeMeters'] - df['AccumulatedDeltaRangeMeters']

        # Absolute distance bias of carrier phase
        meandpc = df.groupby('group_slip')['dpc'].mean()
        df = df.merge(meandpc, on='group_slip', suffixes=('', '_Mean'))

        # Index of original gnss_df
        idx = (gnss_df['Svid'] == svid_sigtype[0]) & (
        gnss_df['SignalType'] == svid_sigtype[1])

        # Carrier phase + bias
        prsmooth[idx] = df['AccumulatedDeltaRangeMeters'] + df['dpc_Mean']

    # If carrier smoothing is not possible, use original pseudorange
    idx_nan = np.isnan(prsmooth)
    prsmooth[idx_nan] = gnss_df['RawPseudorangeMeters'][idx_nan]
    gnss_df['pr_smooth'] = prsmooth

    return gnss_df

In [12]:
# Compute distance by Vincenty's formulae
def vincenty_distance(llh1, llh2):
    """
    Args:
        llh1 : [latitude,longitude] (deg)
        llh2 : [latitude,longitude] (deg)
    Returns:
        d : distance between llh1 and llh2 (m)
    """
    d, az = np.array(pmv.vdist(llh1[:, 0], llh1[:, 1], llh2[:, 0], llh2[:, 1]))

    return d


# Compute score
def calc_score(llh, llh_gt):
    """
    Args:
        llh : [latitude,longitude] (deg)
        llh_gt : [latitude,longitude] (deg)
    Returns:
        score : (m)
    """
    d = vincenty_distance(llh, llh_gt)
    score = np.mean([np.quantile(d, 0.50), np.quantile(d, 0.95)])

    return score

In [13]:
# GNSS single point positioning using pseudorange
def point_positioning(gnss_df):
    # Add nominal frequency to each signal
    # Note: GLONASS is an FDMA signal, so each satellite has a different frequency
    CarrierFrequencyHzRef = gnss_df.groupby(['Svid', 'SignalType'])[
        'CarrierFrequencyHz'].median()
    gnss_df = gnss_df.merge(CarrierFrequencyHzRef, how='left', on=[
                            'Svid', 'SignalType'], suffixes=('', 'Ref'))
    gnss_df['CarrierErrorHz'] = np.abs(
        (gnss_df['CarrierFrequencyHz'] - gnss_df['CarrierFrequencyHzRef']))

    # Carrier smoothing
    gnss_df = carrier_smoothing(gnss_df)

    # GNSS single point positioning
    utcTimeMillis = gnss_df['utcTimeMillis'].unique()
    nepoch = len(utcTimeMillis)
    x0 = np.zeros(4)  # [x,y,z,tGPSL1]
    v0 = np.zeros(4)  # [vx,vy,vz,dtGPSL1]
    x_wls = np.full([nepoch, 3], np.nan)  # For saving position
    v_wls = np.full([nepoch, 3], np.nan)  # For saving velocity
    cov_x = np.full([nepoch, 3, 3], np.nan) # For saving position covariance
    cov_v = np.full([nepoch, 3, 3], np.nan) # For saving velocity covariance

    # Loop for epochs
    for i, (t_utc, df) in enumerate(tqdm(gnss_df.groupby('utcTimeMillis'), total=nepoch)):
        # Valid satellite selection
        df_pr = satellite_selection(df, 'pr_smooth')
        df_prr = satellite_selection(df, 'PseudorangeRateMetersPerSecond')

        # Corrected pseudorange/pseudorange rate
        pr = (df_pr['pr_smooth'] + df_pr['SvClockBiasMeters'] - df_pr['IsrbMeters'] -
              df_pr['IonosphericDelayMeters'] - df_pr['TroposphericDelayMeters']).to_numpy()
        prr = (df_prr['PseudorangeRateMetersPerSecond'] +
               df_prr['SvClockDriftMetersPerSecond']).to_numpy()

        # Satellite position/velocity
        xsat_pr = df_pr[['SvPositionXEcefMeters', 'SvPositionYEcefMeters',
                         'SvPositionZEcefMeters']].to_numpy()
        xsat_prr = df_prr[['SvPositionXEcefMeters', 'SvPositionYEcefMeters',
                           'SvPositionZEcefMeters']].to_numpy()
        vsat = df_prr[['SvVelocityXEcefMetersPerSecond', 'SvVelocityYEcefMetersPerSecond',
                       'SvVelocityZEcefMetersPerSecond']].to_numpy()

        # Weight matrix for peseudorange/pseudorange rate
        Wx = np.diag(1 / df_pr['RawPseudorangeUncertaintyMeters'].to_numpy())
        Wv = np.diag(1 / df_prr['PseudorangeRateUncertaintyMetersPerSecond'].to_numpy())

        # Robust WLS requires accurate initial values for convergence,
        # so perform normal WLS for the first time
        if len(df_pr) >= 4:
            # Normal WLS
            if np.all(x0 == 0):
                opt = scipy.optimize.least_squares(
                    pr_residuals, x0, jac_pr_residuals, args=(xsat_pr, pr, Wx))
                x0 = opt.x 
            # Robust WLS for position estimation
            opt = scipy.optimize.least_squares(
                 pr_residuals, x0, jac_pr_residuals, args=(xsat_pr, pr, Wx), loss='soft_l1')
            if opt.status < 1 or opt.status == 2:
                print(f'i = {i} position lsq status = {opt.status}')
            else:
                # Covariance estimation
                cov = np.linalg.inv(opt.jac.T @ Wx @ opt.jac)
                cov_x[i, :, :] = cov[:3, :3]
                x_wls[i, :] = opt.x[:3]
                x0 = opt.x
                 
        # Velocity estimation
        if len(df_prr) >= 4:
            if np.all(v0 == 0): # Normal WLS
                opt = scipy.optimize.least_squares(
                    prr_residuals, v0, jac_prr_residuals, args=(vsat, prr, x0, xsat_prr, Wv))
                v0 = opt.x
            # Robust WLS for velocity estimation
            opt = scipy.optimize.least_squares(
                prr_residuals, v0, jac_prr_residuals, args=(vsat, prr, x0, xsat_prr, Wv), loss='soft_l1')
            if opt.status < 1:
                print(f'i = {i} velocity lsq status = {opt.status}')
            else:
                # Covariance estimation
                cov = np.linalg.inv(opt.jac.T @ Wv @ opt.jac)
                cov_v[i, :, :] = cov[:3, :3]
                v_wls[i, :] = opt.x[:3]
                v0 = opt.x

    return utcTimeMillis, x_wls, v_wls, cov_x, cov_v

In [14]:
# Simple outlier detection and interpolation
def exclude_interpolate_outlier(x_wls, v_wls, cov_x, cov_v):
    # Up velocity / height threshold
    v_up_th = 2.6  # m/s  2.0 -> 2.6
    height_th = 200.0 # m
    v_out_sigma = 3.0 # m/s
    x_out_sigma = 30.0 # m
    
    # Coordinate conversion
    x_llh = np.array(pm.ecef2geodetic(x_wls[:, 0], x_wls[:, 1], x_wls[:, 2])).T
    x_llh_mean = np.nanmean(x_llh, axis=0)
    v_enu = np.array(pm.ecef2enuv(
        v_wls[:, 0], v_wls[:, 1], v_wls[:, 2], x_llh_mean[0], x_llh_mean[1])).T

    # Up velocity jump detection
    # Cars don't jump suddenly!
    idx_v_out = np.abs(v_enu[:, 2]) > v_up_th
    idx_v_out |= np.isnan(v_enu[:, 2])
    v_wls[idx_v_out, :] = np.nan
    cov_v[idx_v_out] = v_out_sigma**2 * np.eye(3)
    print(f'Number of velocity outliers {np.count_nonzero(idx_v_out)}')

    # Height check
    hmedian = np.nanmedian(x_llh[:, 2])
    idx_x_out = np.abs(x_llh[:, 2] - hmedian) > height_th
    idx_x_out |= np.isnan(x_llh[:, 2])
    x_wls[idx_x_out, :] = np.nan
    cov_x[idx_x_out] = x_out_sigma**2 * np.eye(3)
    print(f'Number of position outliers {np.count_nonzero(idx_x_out)}')

    # Interpolate NaNs at beginning and end of array
    x_df = pd.DataFrame({'x': x_wls[:, 0], 'y': x_wls[:, 1], 'z': x_wls[:, 2]})
    x_df = x_df.interpolate(limit_area='outside', limit_direction='both')

    # Interpolate all NaN data
    v_df = pd.DataFrame({'x': v_wls[:, 0], 'y': v_wls[:, 1], 'z': v_wls[:, 2]})
    v_df = v_df.interpolate(limit_area='outside', limit_direction='both')
    v_df = v_df.interpolate('spline', order=3)

    return x_df.to_numpy(), v_df.to_numpy(), cov_x, cov_v

In [15]:
# Kalman filter
def Kalman_filter(zs, us, cov_zs, cov_us, phone):
    # Parameters
    sigma_mahalanobis = 30.0  # Mahalanobis distance for rejecting innovation

    n, dim_x = zs.shape
    F = np.eye(3)  # Transition matrix
    H = np.eye(3)  # Measurement function

    # Initial state and covariance
    x = zs[0, :3].T  # State
    P = 5.0**2 * np.eye(3)  # State covariance
    I = np.eye(dim_x)

    x_kf = np.zeros([n, dim_x])
    P_kf = np.zeros([n, dim_x, dim_x])

    # Kalman filtering
    for i, (u, z) in enumerate(zip(us, zs)):
        # First step
        if i == 0:
            x_kf[i] = x.T
            P_kf[i] = P
            continue

        # Prediction step
        Q = cov_us[i] # Estimated WLS velocity covariance
        x = F @ x + u.T
        P = (F @ P) @ F.T + Q

        # Check outliers for observation
        d = distance.mahalanobis(z, H @ x, np.linalg.inv(P))

        # Update step
        if d < sigma_mahalanobis:
            R = cov_zs[i] # Estimated WLS position covariance
            y = z.T - H @ x
            S = (H @ P) @ H.T + R
            K = (P @ H.T) @ np.linalg.inv(S)
            x = x + K @ y
            P = (I - (K @ H)) @ P
        else:
            # If observation update is not available, increase covariance
            P += 10**2*Q

        x_kf[i] = x.T
        P_kf[i] = P

    return x_kf, P_kf


# Forward + backward Kalman filter and smoothing
def Kalman_smoothing(x_wls, v_wls, cov_x, cov_v, phone):
    n, dim_x = x_wls.shape

    # For some unknown reason, the speed estimation is wrong only for XiaomiMi8
    # so the variance is increased
    if phone == 'XiaomiMi8':
        v_wls = np.vstack([(v_wls[:-1, :] + v_wls[1:, :])/2, np.zeros([1, 3])])
        cov_v = 1000.0**2 * cov_v
         
    # Forward
    v = np.vstack([np.zeros([1, 3]), (v_wls[:-1, :] + v_wls[1:, :])/2])
    x_f, P_f = Kalman_filter(x_wls, v, cov_x, cov_v, phone)

    # Backward
    v = -np.flipud(v_wls)
    v = np.vstack([np.zeros([1, 3]), (v[:-1, :] + v[1:, :])/2])
    cov_xf = np.flip(cov_x, axis=0)
    cov_vf = np.flip(cov_v, axis=0)
    x_b, P_b = Kalman_filter(np.flipud(x_wls), v, cov_xf, cov_vf, phone)

    # Smoothing
    x_fb = np.zeros_like(x_f)
    P_fb = np.zeros_like(P_f)
    for (f, b) in zip(range(n), range(n-1, -1, -1)):
        P_fi = np.linalg.inv(P_f[f])
        P_bi = np.linalg.inv(P_b[b])

        P_fb[f] = np.linalg.inv(P_fi + P_bi)
        x_fb[f] = P_fb[f] @ (P_fi @ x_f[f] + P_bi @ x_b[b])

    return x_fb, x_f, np.flipud(x_b)

In [17]:
results = []

for i, (group_keys, trip_df) in enumerate(
        tqdm(gnss_test_df.groupby(['drive_id', 'device']))):

    drive, phone = group_keys
    tripID = f'{drive}/{phone}'
    print(tripID)

    # Point positioning
    utc, x_wls, v_wls, cov_x, cov_v = point_positioning(trip_df)

    # Remove velocity outliers
    x_wls, v_wls, cov_x, cov_v = exclude_interpolate_outlier(
        x_wls, v_wls, cov_x, cov_v
    )

    # Kalman smoothing
    x_kf, _, _ = Kalman_smoothing(x_wls, v_wls, cov_x, cov_v, phone)

    assert np.all(~np.isnan(x_kf))

    # build dataframe for this trip
    trip_result = pd.DataFrame({
        "utcTimeMillis": utc,
        "x_kf": x_kf[:, 0],
        "y_kf": x_kf[:, 1],
        "z_kf": x_kf[:, 2],
        "drive_id": drive,
        "device": phone
    })

    results.append(trip_result)

# combine all trips
kfgnss_test_df = pd.concat(results, ignore_index=True)

  0%|          | 0/36 [00:00<?, ?it/s]

2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra


  0%|          | 0/1724 [00:00<?, ?it/s]

Number of velocity outliers 1
Number of position outliers 0
2021-06-22-US-MTV-1/XiaomiMi8


  0%|          | 0/1398 [00:00<?, ?it/s]

...

Number of velocity outliers 12
Number of position outliers 3
2022-04-25-US-OAK-1/GooglePixel5


  0%|          | 0/1912 [00:00<?, ?it/s]

Number of velocity outliers 2
Number of position outliers 1
2022-04-25-US-OAK-2/GooglePixel4


  0%|          | 0/1584 [00:00<?, ?it/s]

Number of velocity outliers 4
Number of position outliers 4


In [19]:
print(kfgnss_test_df)

       utcTimeMillis          x_kf          y_kf          z_kf  \
0      1619650832999 -2.696239e+06 -4.297679e+06  3.852377e+06   
1      1619650833999 -2.696240e+06 -4.297679e+06  3.852377e+06   
2      1619650834999 -2.696240e+06 -4.297679e+06  3.852377e+06   
3      1619650835999 -2.696239e+06 -4.297680e+06  3.852377e+06   
4      1619650836999 -2.696239e+06 -4.297680e+06  3.852377e+06   
...              ...           ...           ...           ...   
66092  1650927742650 -2.671622e+06 -4.292305e+06  3.875412e+06   
66093  1650927743642 -2.671623e+06 -4.292307e+06  3.875414e+06   
66094  1650927744651 -2.671622e+06 -4.292306e+06  3.875412e+06   
66095  1650927745640 -2.671622e+06 -4.292306e+06  3.875412e+06   
66096  1650927746632 -2.671622e+06 -4.292306e+06  3.875412e+06   

                  drive_id                 device  
0      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra  
1      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra  
2      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra  

In [20]:
kfgnss_test = kfgnss_test_df.merge(
    gnss_test_df,
    on=["drive_id","device","utcTimeMillis"],
    how="left"
)

In [22]:
# shrink large gnss file
gnsskf_test_df = kfgnss_test[['utcTimeMillis', 'device', 'drive_id', 'Svid', 'Cn0DbHz', 'MultipathIndicator', 'SvElevationDegrees', 'x_kf', 'y_kf', 'z_kf']]

In [8]:
# gnss_list = []

# for file in gnss_test_files:
#     gnss_file = pd.read_csv(file)
#     parts = file.split(os.sep) # eg ['train', '2021-12-28-US-MTV-1', 'GooglePixel5', 'device_imu.csv']
#     gnss_file["drive_id"] = parts[-3] 
#     gnss_file["device"] = parts[-2]
#     gnss_list.append(gnss_file)

# # combine all
# gnss_test_df = pd.concat(gnss_list, ignore_index=True)
# print(gnss_test_df.head())
# print()
# print(gnss_test_df["device"].unique())
# print()
# print(gnss_test_df["drive_id"].unique())

# # shrink large gnss file
# gnss_test_df = gnss_test_df[['utcTimeMillis', 'device', 'drive_id', 'Svid', 'Cn0DbHz', 'MultipathIndicator', 'SvElevationDegrees', 'WlsPositionXEcefMeters', 'WlsPositionYEcefMeters', 'WlsPositionZEcefMeters']]

  MessageType  utcTimeMillis    TimeNanos  LeapSecond        FullBiasNanos  \
0         Raw  1645655742000  74816000000         NaN -1329690885184087331   
1         Raw  1645655742000  74816000000         NaN -1329690885184087331   
2         Raw  1645655742000  74816000000         NaN -1329690885184087331   
3         Raw  1645655742000  74816000000         NaN -1329690885184087331   
4         Raw  1645655742000  74816000000         NaN -1329690885184087331   

   BiasNanos  BiasUncertaintyNanos  DriftNanosPerSecond  \
0        0.0              5.985788                  NaN   
1        0.0              5.985788                  NaN   
2        0.0              5.985788                  NaN   
3        0.0              5.985788                  NaN   
4        0.0              5.985788                  NaN   

   DriftUncertaintyNanosPerSecond  HardwareClockDiscontinuityCount  ...  \
0                             NaN                                0  ...   
1                         

In [23]:
def get_cn0dbhz(df_gnss):
    df_gnss = df_gnss.rename(columns={"utcTimeMillis": "UnixTimeMillis"})
    
    cn0dbhz_grp_df = df_gnss.groupby(
        ['drive_id', 'device', 'UnixTimeMillis'],
        as_index=False
    )[[
        'Cn0DbHz'
    ]].mean()
    
    return cn0dbhz_grp_df

cn0dbhz_testmean_df = get_cn0dbhz(gnsskf_test_df)

print(cn0dbhz_testmean_df)
print()

                  drive_id                 device  UnixTimeMillis    Cn0DbHz
0      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650832999  27.599822
1      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650833999  27.793549
2      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650834999  27.486261
3      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650835999  27.419616
4      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650836999  28.162419
...                    ...                    ...             ...        ...
66092  2022-04-25-US-OAK-2           GooglePixel4   1650927742650  33.194444
66093  2022-04-25-US-OAK-2           GooglePixel4   1650927743642  32.882353
66094  2022-04-25-US-OAK-2           GooglePixel4   1650927744651  32.980952
66095  2022-04-25-US-OAK-2           GooglePixel4   1650927745640  32.850000
66096  2022-04-25-US-OAK-2           GooglePixel4   1650927746632  32.785714

[66097 rows x 4 columns]



In [24]:
def get_satcount(df_gnss):
    df_gnss = df_gnss.rename(columns={"utcTimeMillis": "UnixTimeMillis"})
    
    satcount_grp_df = df_gnss.groupby(['drive_id', 'device', 'UnixTimeMillis'], as_index=False)['Svid'].count()
    
    satcount_grp_df = satcount_grp_df.rename(columns={'Svid': 'SatCount'})
    return satcount_grp_df


def merge_target(merge_df, target_df):
    
    df_merge = merge_df.merge(target_df[['UnixTimeMillis',  'drive_id', 'SatCount', 'device']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'inner')
    
    return df_merge

satcount_testgrp_df = get_satcount(gnsskf_test_df)
merge_test_df = merge_target(cn0dbhz_testmean_df, satcount_testgrp_df)
print(merge_test_df)
print()

                  drive_id                 device  UnixTimeMillis    Cn0DbHz  \
0      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650832999  27.599822   
1      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650833999  27.793549   
2      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650834999  27.486261   
3      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650835999  27.419616   
4      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650836999  28.162419   
...                    ...                    ...             ...        ...   
66092  2022-04-25-US-OAK-2           GooglePixel4   1650927742650  33.194444   
66093  2022-04-25-US-OAK-2           GooglePixel4   1650927743642  32.882353   
66094  2022-04-25-US-OAK-2           GooglePixel4   1650927744651  32.980952   
66095  2022-04-25-US-OAK-2           GooglePixel4   1650927745640  32.850000   
66096  2022-04-25-US-OAK-2           GooglePixel4   1650927746632  32.785714   

       SatCount  
0            39  
1  

In [25]:
def get_elevationdeg(df_gnss):
    df_gnss = df_gnss.rename(columns={"utcTimeMillis": "UnixTimeMillis"})
    
    elevationdeg_grp_df = df_gnss.groupby(
        ['drive_id', 'device', 'UnixTimeMillis'],
        as_index=False
    )[[
        'SvElevationDegrees'
    ]].mean()
    
    return elevationdeg_grp_df

def merge_target(merge_df, target_df):
    
    df_merge = merge_df.merge(target_df[['UnixTimeMillis',  'drive_id', 'SvElevationDegrees', 'device']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'inner')
    
    return df_merge

elevationdeg_testgrp_df = get_elevationdeg(gnsskf_test_df)
merge_test_df = merge_target(merge_test_df, elevationdeg_testgrp_df)
print(merge_test_df)
print()

                  drive_id                 device  UnixTimeMillis    Cn0DbHz  \
0      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650832999  27.599822   
1      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650833999  27.793549   
2      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650834999  27.486261   
3      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650835999  27.419616   
4      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650836999  28.162419   
...                    ...                    ...             ...        ...   
66092  2022-04-25-US-OAK-2           GooglePixel4   1650927742650  33.194444   
66093  2022-04-25-US-OAK-2           GooglePixel4   1650927743642  32.882353   
66094  2022-04-25-US-OAK-2           GooglePixel4   1650927744651  32.980952   
66095  2022-04-25-US-OAK-2           GooglePixel4   1650927745640  32.850000   
66096  2022-04-25-US-OAK-2           GooglePixel4   1650927746632  32.785714   

       SatCount  SvElevationDegrees  
0

In [26]:
elevationdeg_mean = merge_test_df['SvElevationDegrees'].mean()
print('SvElevationDegrees Mean: ' + str(elevationdeg_mean))

merge_test_df['SvElevationDegrees'] = merge_test_df['SvElevationDegrees'].fillna(elevationdeg_mean)

SvElevationDegrees Mean: 37.98593473680146


In [27]:
def get_tripid(df):
    df['tripId'] = df['drive_id'] + '/' + df['device']
    df.drop(columns=['drive_id', 'device'], inplace=True)
    
    return df

merge_test_df = get_tripid(merge_test_df)
imu_testacc_df = get_tripid(imu_testacc_df)
imu_testgyro_df = get_tripid(imu_testgyro_df)
imu_testmag_df = get_tripid(imu_testmag_df)
gnsskf_test_df = get_tripid(gnsskf_test_df)

/var/folders/w8/d9x92py92j792gw_1kyfc8sh0000gn/T/ipykernel_37730/559178080.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tripId'] = df['drive_id'] + '/' + df['device']
/var/folders/w8/d9x92py92j792gw_1kyfc8sh0000gn/T/ipykernel_37730/559178080.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns=['drive_id', 'device'], inplace=True)


In [28]:
def get_accmag(merge_df):
    merge_df = merge_df.rename(columns={"utcTimeMillis": "UnixTimeMillis"})
    
    merge_df['AccMag'] = np.sqrt(merge_df['MeasurementX']**2 + merge_df['MeasurementY']**2 + merge_df['MeasurementZ']**2)
    
    return merge_df

def merge_target(merge_df, target_df):
    df_gt = target_df.sort_values('UnixTimeMillis')
    df_acc = merge_df.sort_values('utcTimeMillis')
    df_acc = df_acc.rename(columns={"utcTimeMillis": "UnixTimeMillis"})

    final_df = pd.merge_asof(
        df_gt, 
        df_acc, 
        on='UnixTimeMillis', 
        by='tripId', 
        direction='nearest',
        tolerance=50 
    )
    
    final_df = final_df.sort_values(['tripId', 'UnixTimeMillis']).reset_index(drop=True)
    
    return final_df

acc_testmerge_df = merge_target(imu_testacc_df, merge_test_df)
acc_testmerge_df = get_accmag(acc_testmerge_df)
acc_testmerge_df.drop(columns=['MeasurementX', 'MeasurementY', 'MeasurementZ'], inplace=True)
merge_test_df = acc_testmerge_df
print(merge_test_df)

       UnixTimeMillis    Cn0DbHz  SatCount  SvElevationDegrees  \
0       1619650832999  27.599822        39           38.017116   
1       1619650833999  27.793549        39           38.015823   
2       1619650834999  27.486261        39           37.268639   
3       1619650835999  27.419616        39           37.267143   
4       1619650836999  28.162419        39           38.337918   
...               ...        ...       ...                 ...   
66092   1650927742650  33.194444        18           34.795333   
66093   1650927743642  32.882353        17           35.171023   
66094   1650927744651  32.980952        21           35.621820   
66095   1650927745640  32.850000        20           37.707205   
66096   1650927746632  32.785714        21           35.621205   

                                          tripId    AccMag  
0      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra  9.759161  
1      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra  9.854152  
2      2021-04-28-US-MTV

In [29]:
def get_gyromag(merge_df):
    merge_df = merge_df.rename(columns={"utcTimeMillis": "UnixTimeMillis"})
    
    merge_df['GyroMag'] = np.sqrt(merge_df['MeasurementX']**2 + merge_df['MeasurementY']**2 + merge_df['MeasurementZ']**2)
    
    return merge_df

def merge_target(merge_df, target_df):
    df_gt = target_df.sort_values('UnixTimeMillis')
    df_gyro = merge_df.sort_values('utcTimeMillis')
    df_gyro = df_gyro.rename(columns={"utcTimeMillis": "UnixTimeMillis"})

    final_df = pd.merge_asof(
        df_gt, 
        df_gyro, 
        on='UnixTimeMillis', 
        by='tripId', 
        direction='nearest',
        tolerance=50 
    )
    
    final_df = final_df.sort_values(['tripId', 'UnixTimeMillis']).reset_index(drop=True)
    
    return final_df

gyro_testmerge_df = merge_target(imu_testgyro_df, merge_test_df)
gyro_testmerge_df = get_gyromag(gyro_testmerge_df)
gyro_testmerge_df.drop(columns=['MeasurementX', 'MeasurementY', 'MeasurementZ'], inplace=True)
merge_test_df = gyro_testmerge_df
print(merge_test_df)

       UnixTimeMillis    Cn0DbHz  SatCount  SvElevationDegrees  \
0       1619650832999  27.599822        39           38.017116   
1       1619650833999  27.793549        39           38.015823   
2       1619650834999  27.486261        39           37.268639   
3       1619650835999  27.419616        39           37.267143   
4       1619650836999  28.162419        39           38.337918   
...               ...        ...       ...                 ...   
66092   1650927742650  33.194444        18           34.795333   
66093   1650927743642  32.882353        17           35.171023   
66094   1650927744651  32.980952        21           35.621820   
66095   1650927745640  32.850000        20           37.707205   
66096   1650927746632  32.785714        21           35.621205   

                                          tripId    AccMag   GyroMag  
0      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra  9.759161  0.007877  
1      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra  9.854152  0.013

In [16]:
# def get_magmag(merge_df):
#     merge_df = merge_df.rename(columns={"utcTimeMillis": "UnixTimeMillis"})
    
#     merge_df['MagMag'] = np.sqrt(merge_df['MeasurementX']**2 + merge_df['MeasurementY']**2 + merge_df['MeasurementZ']**2)
    
#     return merge_df

# def merge_target(merge_df, target_df):
#     df_gt = target_df.sort_values('UnixTimeMillis')
#     df_mag = merge_df.sort_values('utcTimeMillis')
#     df_mag = df_mag.rename(columns={"utcTimeMillis": "UnixTimeMillis"})

#     final_df = pd.merge_asof(
#         df_gt, 
#         df_mag, 
#         on='UnixTimeMillis', 
#         by='tripId', 
#         direction='nearest',
#         tolerance=500 
#     )
    
#     final_df = final_df.sort_values(['tripId', 'UnixTimeMillis']).reset_index(drop=True)
    
#     return final_df

# mag_testmerge_df = merge_target(imu_testmag_df, merge_test_df)
# mag_testmerge_df = get_magmag(mag_testmerge_df)
# mag_testmerge_df.drop(columns=['MeasurementX', 'MeasurementY', 'MeasurementZ'], inplace=True)
# merge_test_df = mag_testmerge_df
# print(merge_test_df)

In [17]:
# def get_energy(merge_df):
#     merge_df['TotalMotionEnergy'] = np.sqrt(merge_df['AccMag']**2 + merge_df['GyroMag']**2 + merge_df['MagMag']**2)

#     return merge_df

# merge_test_df = get_energy(merge_test_df)

In [30]:
def get_satcount_delta(merge_df):
    merge_df['SatCountDelta'] = merge_df['SatCount'].diff().fillna(0)

    return merge_df

merge_test_df = get_satcount_delta(merge_test_df)

In [31]:
# def get_energy_delta(merge_df):
#     merge_df['EnergyDelta'] = merge_df['TotalMotionEnergy'].diff().fillna(0)

#     return merge_df

# merge_test_df = get_energy_delta(merge_test_df)

In [32]:
def get_cn0dbhz_delta(merge_df):
    merge_df['Cn0DbHzDelta'] = merge_df['Cn0DbHz'].diff().fillna(0)

    return merge_df

merge_test_df = get_cn0dbhz_delta(merge_test_df)

In [33]:
def get_cn0dbhz_mean(merge_df):
    merge_df['Cn0DbHzMean'] =  merge_df['Cn0DbHz'].rolling(window=5).std().fillna(0)

    return merge_df

merge_test_df = get_cn0dbhz_mean(merge_test_df)

In [34]:
# def get_energy_std(merge_df):
#     merge_df['EnergyStd'] =  merge_df['TotalMotionEnergy'].rolling(window=5).std().fillna(0)

#     return merge_df


# merge_test_df = get_energy_std(merge_test_df)

In [35]:
# def get_signal_clarity(merge_df):
#     merge_df['SignalClarity'] =  merge_df['Cn0DbHz'] * merge_df['SatCount']
#     return merge_df

# merge_test_df = get_signal_clarity(merge_test_df)

In [37]:
def get_wls_val(df_gt, df_gnss):
    gnss_grp_df = df_gnss.groupby(
        ['tripId', 'utcTimeMillis'],
        as_index=False
    )[[
        'WlsPositionXEcefMeters',
        'WlsPositionYEcefMeters',
        'WlsPositionZEcefMeters'
    ]].first()
    
    gnss_renamed_df = gnss_grp_df.rename(columns={"utcTimeMillis": "UnixTimeMillis"})

    df_merge = df_gt.merge(gnss_renamed_df[['UnixTimeMillis', 'WlsPositionXEcefMeters', 'WlsPositionYEcefMeters', 'WlsPositionZEcefMeters', 'tripId']], on = ['UnixTimeMillis', 'tripId'], how = 'left')
    
    return df_merge 

def get_kf_val(df_gt, df_gnss):
    gnss_grp_df = df_gnss.groupby(
        ['tripId', 'utcTimeMillis'],
        as_index=False
    )[[
        'x_kf',
        'y_kf',
        'z_kf'
    ]].first()
    
    gnss_renamed_df = gnss_grp_df.rename(columns={"utcTimeMillis": "UnixTimeMillis"})

    df_merge = df_gt.merge(gnss_renamed_df[['UnixTimeMillis', 'x_kf', 'y_kf', 'z_kf', 'tripId']], on = ['UnixTimeMillis', 'tripId'], how = 'left')
    
    return df_merge 

# get wls_val 
# merge_test_df = get_wls_val(merge_test_df, gnss_test_df)

# get kf_val
merge_test_df = get_kf_val(merge_test_df, gnsskf_test_df)
print(merge_test_df)

       UnixTimeMillis    Cn0DbHz  SatCount  SvElevationDegrees  \
0       1619650832999  27.599822        39           38.017116   
1       1619650833999  27.793549        39           38.015823   
2       1619650834999  27.486261        39           37.268639   
3       1619650835999  27.419616        39           37.267143   
4       1619650836999  28.162419        39           38.337918   
...               ...        ...       ...                 ...   
66092   1650927742650  33.194444        18           34.795333   
66093   1650927743642  32.882353        17           35.171023   
66094   1650927744651  32.980952        21           35.621820   
66095   1650927745640  32.850000        20           37.707205   
66096   1650927746632  32.785714        21           35.621205   

                                          tripId    AccMag   GyroMag  \
0      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra  9.759161  0.007877   
1      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra  9.854152  0.0

In [38]:
# wlsx_mean = merge_test_df['WlsPositionXEcefMeters'].mean()
# print('WlsPositionXEcefMeters Mean: ' + str(wlsx_mean))
# wlsy_mean = merge_test_df['WlsPositionYEcefMeters'].mean()
# print('WlsPositionYEcefMeters Mean: ' + str(wlsy_mean))
# wlsz_mean = merge_test_df['WlsPositionZEcefMeters'].mean()
# print('WlsPositionZEcefMeters Mean: ' + str(wlsz_mean))

# merge_test_df['WlsPositionXEcefMeters'] = merge_test_df['WlsPositionXEcefMeters'].fillna(wlsx_mean)
# merge_test_df['WlsPositionYEcefMeters'] = merge_test_df['WlsPositionYEcefMeters'].fillna(wlsy_mean)
# merge_test_df['WlsPositionZEcefMeters'] = merge_test_df['WlsPositionZEcefMeters'].fillna(wlsz_mean)

xkf_mean = merge_test_df['x_kf'].mean()
print('x_kf Mean: ' + str(xkf_mean))
ykf_mean = merge_test_df['y_kf'].mean()
print('y_kf Mean: ' + str(ykf_mean))
zkf_mean = merge_test_df['z_kf'].mean()
print('z_kf Mean: ' + str(zkf_mean))

merge_test_df['x_kf'] = merge_test_df['x_kf'].fillna(xkf_mean)
merge_test_df['y_kf'] = merge_test_df['y_kf'].fillna(ykf_mean)
merge_test_df['z_kf'] = merge_test_df['z_kf'].fillna(zkf_mean)

x_kf Mean: -2624538.64244257
y_kf Mean: -4435676.420544275
z_kf Mean: 3736536.644763804


In [39]:
print(merge_test_df.isna().sum())

UnixTimeMillis        0
Cn0DbHz               0
SatCount              0
SvElevationDegrees    0
tripId                0
AccMag                0
GyroMag               0
SatCountDelta         0
Cn0DbHzDelta          0
Cn0DbHzMean           0
x_kf                  0
y_kf                  0
z_kf                  0
dtype: int64


In [27]:
# merge_test_df.drop(columns=['AccMag', 'GyroMag', 'MagMag'], inplace=True)

In [40]:
%store merge_test_df

Stored 'merge_test_df' (DataFrame)


In [29]:
test_check = pd.DataFrame()

test_check['tripId'] = merge_test_df['tripId']

%store test_check

Stored 'test_check' (DataFrame)
